# Week2 作业：商场客户聚类与营销策略

**数据集**：`Mall_Customers.csv`  
**工具**：Pandas、Matplotlib/Seaborn、Scikit-learn  

任务：EDA → K-Means 聚类 → 分群画像 → 营销建议

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler
from IPython.display import display

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

WEEK2 = Path('.').resolve()
FIG = WEEK2 / 'figures'
FIG.mkdir(exist_ok=True)

INCOME = 'Annual Income (k$)'
SPEND = 'Spending Score (1-100)'

## 任务 1：数据探索（EDA）

In [ ]:
df = pd.read_csv(WEEK2 / 'Mall_Customers.csv')
print('规模:', df.shape)
display(df.head())
print('\n字段类型:')
print(df.dtypes)
print('\n缺失值:')
print(df.isna().sum())
print('\n描述统计:')
display(df.describe(include='all'))

In [ ]:
# 异常值（IQR）
for col in ['Age', INCOME, SPEND]:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    n = ((df[col] < q1 - 1.5*iqr) | (df[col] > q3 + 1.5*iqr)).sum()
    print(f'{col}: IQR 异常 {n} 条')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
sns.histplot(df['Age'], kde=True, ax=axes[0])
sns.histplot(df[INCOME], kde=True, ax=axes[1])
sns.histplot(df[SPEND], kde=True, ax=axes[2])
plt.tight_layout(); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
sns.boxplot(data=df, x='Gender', y='Age', ax=axes[0])
sns.boxplot(data=df, x='Gender', y=SPEND, ax=axes[1])
plt.tight_layout(); plt.show()

sns.scatterplot(data=df, x=INCOME, y=SPEND, hue='Gender')
plt.title('年收入 vs 消费评分'); plt.show()

## 任务 2：K-Means 聚类

In [ ]:
X = df[[INCOME, SPEND]].values
scaler = StandardScaler().fit(X)
Xs = scaler.transform(X)

rows = []
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    lab = km.fit_predict(Xs)
    rows.append({'k': k, 'inertia': km.inertia_, 'silhouette': silhouette_score(Xs, lab)})
metrics = pd.DataFrame(rows)
display(metrics)

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(metrics.k, metrics.inertia, 'bo-'); ax[0].set_title('手肘法')
ax[1].plot(metrics.k, metrics.silhouette, 'ro-'); ax[1].set_title('轮廓系数')
plt.show()

best_k = int(metrics[metrics.k.isin([4,5,6])].sort_values('silhouette', ascending=False).iloc[0]['k'])
print('选定 k =', best_k)

In [ ]:
km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df['Cluster'] = km.fit_predict(Xs)
centers = scaler.inverse_transform(km.cluster_centers_)

plt.figure(figsize=(8, 6))
sns.scatterplot(data=df, x=INCOME, y=SPEND, hue='Cluster', palette='tab10', s=80)
plt.scatter(centers[:,0], centers[:,1], c='black', marker='X', s=200, label='中心')
plt.legend(); plt.show()
df.to_csv(WEEK2 / 'output' / 'Mall_Customers_clustered.csv', index=False)

## 任务 3 & 4：分群画像与营销建议

运行 `python run_analysis.py` 可生成完整画像表与 `marketing_strategies.csv`。

In [ ]:
prof = df.groupby('Cluster').agg(
    人数=('Cluster', 'count'),
    平均年龄=('Age', 'mean'),
    平均年收入_k=(INCOME, 'mean'),
    平均消费评分=(SPEND, 'mean'),
)
prof['男性占比_%'] = df.groupby('Cluster')['Gender'].apply(lambda s: (s=='Male').mean()*100)
display(prof.round(1))